# Part 3 — Can we trust the model forever? (CIFAR-10-C)

Raw confidence of the frozen ResNet-18 used in Parts 2 and 4, on clean CIFAR-10 and CIFAR-10-C severities 1–5
(Gaussian noise, motion blur, brightness, contrast, pixelate).

*Runtime → Change runtime type → T4 GPU*, then *Run all*. Most of the time goes into downloading CIFAR-10-C (2.9 GB).
Set `USE_DRIVE = True` to keep the downloads and logits in Google Drive, so later runs take about a minute.

In [ ]:
REPO_URL = "https://github.com/puppyvn/neural_network_confidence.git"
BRANCH = "main"                # branch to clone; falls back to the default branch if it does not exist
USE_DRIVE = False

import os, io, base64, zipfile, subprocess
os.chdir("/content" if os.path.isdir("/content") else os.getcwd())
if not os.path.exists("neural_network_confidence"):
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL]).returncode != 0:
        print(f"Branch {BRANCH!r} not found; cloning the default branch.")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

# Part 3 code, embedded so this also works before the folder is pushed to the branch.
PART3_CODE_ZIP = (
    "UEsDBBQAAAAIAIsYR13kXGeO2AYAAFYQAAAZAAAAcGFydF8zL2NpZmFyMTBjL2NvbW1vbi5weY1XbXMatxb+zq/Q3XzZbZctYIe8"
    "9NIpIbTjqWt7HNKZOwyzIxYBul4kVdImxp32t/ccSftih3jCB1hpj87L8zw6ElEUfdhTzTbEMGu52JmUHJjVvDCEig1RpbRkz0rF"
    "tCFbqckN1ZackVjTz6SQYss3TBSMVGLDNJld/DK97Q8H/Rkxe761SRZFUW+r5YEoavclXxN+UBI83MCwF55FdVBHQg0Rytuagqtj"
    "Ziy1prbXVNxtqKW93m8peXdx9YFMyHCQkuHL3uz69vbjzeLi2k3G0Y5WxnAqciG5YVFKooO0XIp8XVYah2vNd3srmDE4giKspsbi"
    "s+L3rKSWRUnvw/yP+e3F4mLunA5TMkrJWUrOU/Iy6f0+n17h9CA7fzOEKfh9PRq53/MxvP+weO9fj85fDXF6dH720v2Oh+Ok937+"
    "y/Tj5SK/vb5egB1iEef5lpcsz5NMMyPLTyxOMgXECEt+IBGAzre0sCbq9W6ml/PFYg4L/4oK4K6QtIzekujFmfu4CsuKuanRcDye"
    "znBKAoI7P/l+PB4MplHaI+0n2mnGhHs9fPfq9dkrXAOycDOzwdmb0TsHUKVV6b2Mp2fv30yjv3svyKxkVEA+dTYpsXsmiBQMJFLK"
    "ShOQDwjsE9PcHkls6KF+Y5B3sPa6OidbvqsAgCSrCfhfPru+vL5FFja8sPEDV3E8SBPyPWkpelTK088yALYsVk7BBeECZNJAl9bF"
    "18C1aAUM2sKTVZJ0BHcit44a06+Fbr0Ht53IIZ7PyIXr9TZsS5SW/2eFzXEfmVhLaZO3rmp8rEXkpjN2r2DrVobpuKMmZ1zQYs9S"
    "ApNVCZtr4leDwNwLCFyPg0XkVmHqW1niDsf80SZ94iok463RMjvcbbiOvYTNZKErMGb33Nhc3rmhz0gzW2lBTvkMlRu5tQd6Hz+E"
    "GA+QtlAZNVRreowfUrKxR8UmMLctJbXj86SxeyB98pDhagqhJ7AX7xhTG34wnRSYdwiwQYxuUgyQYJmpDl9b7RPUrOR0zUvQdr7m"
    "wsQqJceUCDeYYLMKmUMzjA+4VdrGmRJaFJWmBSwoZCVsQuSW4E45EiFFnx0U7Bj2Z0XL/me+sXsCTl1XdXSCH0hedSoMPEutQS3Y"
    "hAApveu8J5MJOSYAH4IWO8T8GnCc843H4sAFP0DdsYvwXailWcUhz7pAQDgEVRLmUVTLVSObNSrGyT0OPlqhGFZCjgxD1rHhqXnN"
    "t41FRsUx7qxso2VUKSY2PtFlbb/KEOY4SWsgTrxpfCO/SfKI+I68fJhaYwEuzzvdUGX5J5ZbqfKSrln5HPPT2fwtAcu+swSpw2mo"
    "KR5MhGkNUEkgPTB9oMYQB258xe/RwhJaZnAIDYZvktP0P8P8M6TvtKyUCTsKC86NAiHHbrgzcPQ6ZEH5XGwmEZzJa2yDdY2PUEMg"
    "SybiXQL7Bh+OCUiHro0nZ1djD4qpaWnmEqeWHaolpAT0e2cN3AXLixKQ+QzH+leRdlg+EeFdK0KVmT1VbDlcdeT0DSCo5VvAYPU8"
    "Dm18aCYWZYnFq+UOl7bFx0eoGxm5S76l+g7CjjhMyy3zoRp8NDd3eYEqojuPT0d8RcGUdQftHTlIY5sWZOFkYXh6gRBB5nd4r8qy"
    "qx9DSFBg7TJ1EVr1SY0HQgDNo9SvpfgUp8fUdJW5dG5W5D8g0PB8SqfUFcA2dTxHJjTkILPv6x4UgGrMayGmuKyoDqjRgBu8q81q"
    "hVVaFnnQZtB9GLVQ3sQUtbSBe2q911oECTcAr2YdeO0eGn6z5LOWYoc3o+RHuA6+xHIk0A70H1wv6Ozteh93mlGYrLvRWsrSly0A"
    "CjHAK5i38E0tJfE/df5+phcaqxii/gbAoVsGj+1ueKS1SFAR6EOu62u4A+eENGO9DAFXPiDIHWJ9R2L4RpIA9BF+xW5WDBr1sk+0"
    "rODq3RUuPj4GQHki6+ExiMPAXQdo8Hsby4lr1n9L3N+YIJOfyKC7ADxxs4WjDsOC6soy9uaxIj8BJt0pjFqWRSmNayJQG6r86cl3"
    "8irgbXK/f9Doy43atPO0Q3y7mb5s5V3o/2qow33uLhNwQ/eE1HoIfaa9KEfz2byxEr4iXwZ2utGq28KxdmxkzevBCnhth9BMH7me"
    "hoNxIdUlHnZNnK+cmI8Xz+Ztm4eVX7b9rvXV5WXjvQ95lnKH1RQlXMXVsm0VvhaA8gg9fMj6ZwP885gkJ4C5BXp+fj1o3CJdS8Qe"
    "VkDFzj3jZZy9BmiC38TdglZfuBl/g5vxs25+h/xmIIwOo2J7IutfqTpl0p61zRq/4AVuBnfnaFpVSv4LU+6ffDPX4eXj7fUM+Xim"
    "Sfp8/u79C1BLAwQUAAAACACXGEddzr4/qy8LAABRHAAAHwAAAHBhcnRfMy9jaWZhcjEwYy9wcmVwYXJlX2RhdGEucHmVWXtz1DgS"
    "/38+hU5bV2XXejSTQBaY27krLglLallgQ/YBqZRLseUZg19IcpKB4rtfd0se25MEuClIxrLU3ernrzuc86P6uipqmTK7VizT9SdV"
    "sVNlXio73XscscOTZ09Pp3tzJqt0+zA9jOg5kQkcwoNFvcqtYa+ltuwBq5RKjZhMXtZ2nVcrlhtmtcwrlQp2BrvhVPKhqfPK4qt1"
    "u7mqmrWsZq83Z7VO1jHxAZ4DOZCJkaVidaVYa1TK8mqC7AzbJ1kesuDJAzF/9E+WFEpWTCZJq2WyCQU7bSs4lqh/sUJapZluK8O0"
    "AipElm4B0jL4NBu7rivWaNVIreJUWimaDbv78wNLvfIMU1dKb/xtK1szMQPZ8kwm1nyN8HSq69qyWSPtembrWZprIqzESjDJfqnr"
    "VaHYkc6vwDh1kSo9mZxkvTayvFCGyUIrmW6YusmNRSOWzhKgE+Bm44czo5NeoBndOIyYM4GXQdrJXZsXvY5AAl1KslmeqsrmiSxI"
    "91Vv504jYCC5ApOLCed8kpdNDfJIvQIWRnXPa2nWRX7ZPbpfsCBamxfd6ntTV913sx6+Me1lo+tEGdOtWKlRI93jp7wZPlZtCSqX"
    "hlXNhJSU1GUJRvGvfwVvf3V6+sfrs5NXL99E7Lfjpy8j9ubsKGJHx8+e/vHiLD599eoM1Kbr9yqxMRrNTCZ/HZ/88vzsTfzL0enJ"
    "n8fxyRFbMr73KCtfPla/py+aj/vvT377Pf57fvL30eu/s5Mn9Z9/rT7yiffyw/iP0xd4ZG1tYxazGURgndai1quZVkmtUzPbP3hw"
    "8OSnRzMy96wPQwEX/k+n8uUen0wmqcoYPsVlnaoiQNNGznwRS9VVnqhwQR7ZqQwjbriwQnq0AL5ag2DOQxnfiU7uDmVofRbQXtiU"
    "gNfovbnjbjguaWUqZcHbeSjIQ03gRcBPb0RBaSEGpyqCcw75hEdAr4CAxy/Taaoau8avezyasK9/tsoEMuv2UoClZ/fmGeF4Gavp"
    "GuFFSOTRO3ILCrglYt3aprW9kNPD4fEIr3w1JU/Ht8+Pnx7xC0hh6sYuz3SrQgF78yZwbK5Vvlojm6+o0FhIXHGaQzwOVLr3WDR2"
    "ZAZH6w4tS7gu5pClj+QxTQGBwrd7PS0fPCI3sf8aeCoDuvixZXMf3c41ITmMeXg+5GvbXUGeLm8FU8SctpeoX2AF6v3Y5soun8nC"
    "qBBzzkusCbW+T2o8tLjlMFCPIP9DabB5qY61rnXA/3Km6ORhmYTzULO2NXJHZy7VYn7crVz8Ww6Kn4yfgmv8dkw5FK7IcszC7LNX"
    "8xfBw11FC3CSQia9KbYbrsHRt7d/lzfPhgbDpPcpGyuhwvqxZOcVpnUGWbCCLQJXC/CeICQ/EKpKDZIORj4XXpzPL0bUvOdhcaus"
    "KD9AIQvcgyGfj1xxiusPLgTGh0n2TNSNqgKUgASGMhR1dOkNv77k9CY19rZBXW2ASG82qIP68n1AFGCz42bWEu7ra46Ap/2DnwJP"
    "H+tnfLmxCqImFGt1k+YrhVpwJxuVwNFxfRK4GqMHxMgvLupE2rwGMZO6yrBAJip2SoMs8H35kbjBq7ZQt/m5dccReQf4oxdQUNnV"
    "EPwgltsbuF9bsqoAqm5NdOYMAJJ4cOZDqt9NJOPe6QMqFrTqFRexUjbbqy950rS8M5qJ66rYOHPfEgGI2zrwBUmoK1kEIcgEoQ2C"
    "xSsNfIOBOJB/r1QlQaVwHBGBSKGYm+Az77EkX6CJsWZQ4oZH9wVWSkCE8OwqOmSoFB6gsH/piMPpgG/R5hCfvnn+dAp+ssAEv5ah"
    "r4u21ZW7RTQQzVdfgJe++PodeSnBmeIW6D3uinDELqWFhGHyT10+BaDkdi5YAGI+2Pf/Q0Yn2fTfHczG13vzkGVgCfvTw8hhQcSW"
    "rkwhFpMOjW/hsUAcdmfdR+WAVp1traoMpELUVCiucnUd7IEIwA7+eV+z6e5uUOU9myF3Y45xuYLC3B3Mq0xpihDU0bBSYTICj0M4"
    "WEGSrlYqmEesgPAfqjG8Q3/d52YrHoUKob6gaoTUWm5GVM6J0cKx+3FAEao15pGh7w6p34hG6bK1JJq7734oyBqQN2ds/+BgdAa0"
    "IGQDKSx1LhEEwQ2bkuJxO2g0HERDKCCKIBqc4OHI5+AakF0g2sDjgD0QDoU0dtMovKH3h9A7IpXkNECYOnRT0Ka8hOSzxLrZgUFX"
    "83HrHejBM8f9vSGBHyUCRx5AW30dN3nyoVBdacYcLq9UuhhCCyzjtHrOe5H4Rcj+sRzIOLboLn9Pyl0DEQDKTijAdyOsM3cMGQVS"
    "i2foDiAWc9/Cb7Pxax0Bij8ksEPRa7yEPNopTiIq6joe8VSv2hKq4Wt80mBqkwAEpKQZx2mdxHHk2yvoUOOkkMYst4dP5fVRf+C5"
    "Kppn3dbQ8xIyTWPpmQTc9XQcs00m28Iuxx3MGkgs+TNqKCni+kYWNbjt6J0L8XuZOI8FNsm6hi9mGXDZ2pqAO1UCnrSp5GEvh3t/"
    "L0EKwSmGIBxGt15CnPanD+Zzf1SvDKq3EaQhpAEeO/lqY0MLV7kBFQ6XsehAqXdtugN0H9NSoJzbk7BAr919gXHgLoZe6FINPiLo"
    "lFeAGeUl1N6QgWcopwgCUyij6ChAv0eacJsGr4YV6YhWFnzbublUMGzpYrYc96MB0cI9g5objQvoN/pDp8Yf2CENUgATQautYFed"
    "ZXmSQ88PPSkADdckFYPq4ZQrcLgBB4zwaJiYEH7vWkncgV5BwMOli2jrgwOA+BYoYygbl7uJGfa8K6AOBzoHcZvX1Kz5pLdtR+hM"
    "7AJXVM0nPtRF2CU/POs7iT4nvMOmbLeYOxnwAn0pJ5UP6tGWAsiOieJTDEgEyrMx35DMPSzfbVP026G0y13B0XfG4sI9OmTufCjj"
    "h6Nx2IJ9Dt5BblyV8ibYC9ER34YCKxHkLfEw+8JYoG4ATlqVsrl48mD+KOS9R2wnDwuG4M4NDbGxhK6kAggJZVPrltIUpBKtAPWD"
    "jYnYtlfyTYn3nxhHXv2QgTwDv/SseL/xexsLHEAqhCnnGf+cfAHtbjjluQRRxWDKcwFlv8vitMvpztYpTj7OkzsPocPsuFnG+4vH"
    "juPY0yJQc+dhF53XEZuuZOHYI3DqmLGqH5X07Zm71de7+vFs6FZT3+n+do2/o5Mf0Bo28iO6WzjQpVEBsRkMR1s0GVGy9NbCdhtg"
    "yzJ4ANhp7/E8JJyg72jQBbXoMVyfWpDWBOGtTba2EjsKdHUNfRt2QE4EflhXgJLs9IWqVjQ3QlagylF5H10B++txo5lFlPsDYrOk"
    "nxFrqxzq2H+5+xabRALkcZfDsr4cui5RuZR3XK/Dusm6rT4Q1hU5lX4ndvCQ/fwz25+Hi3snCZm41nAkIArhvduAu2ibFOEiomi3"
    "+/8dLPjBqlPPcKpgd6YKeKVSlZeAK+BONkNbuOdb3jYYQrgdNHwQ2jRFDvabcewizqd7F7dOgTP7E7mhEU/oogiJbeNkG1h9UOFk"
    "4R7XH982Ez5vEXXHq59J7BD8jtnEd88ofFLoKbjUFCeuEBLg7rgP01ZvLgCOSt8CwB2ZCF9YvNRb11R1L6APmdHC2zDECWbvwx3M"
    "hlSF2iyxzAmXAO5KjiOwTy/HF3LIC3uwO67UZ+uIlThZwKK75HpnEObv6MjgJKchRBUAPJzPh53z6JRvnpf3Nej313McliCT4AD7"
    "buLx65j21wr9PbXB13r36zsLfvdxtT1xs44GGtF0oKExKtjZTjLR3766Ix5rvujxPnN/Tcor7pEhNpRgzThGh49jQq9xjM1OHHPH"
    "ynU+k/8BUEsDBBQAAAAIAKcYR13JP7+jYAsAACscAAAcAAAAcGFydF8zL2NpZmFyMTBjL3J1bl9wYXJ0My5weZ1Z627jxhX+r6eY"
    "sjB2uJG4ku1NN0qZ1tHazSKb2LCdALuqQIzIocSYt3JIW1rDQN6hfcI8Sb8zvEvyJq1gWOTMmXO/jgzDuBJZzk7YiM1EzB4ky7NC"
    "5SxfSxYlngyZn2TyXmZ/Y9figblJ7AeejF3JitiTGQtiN5NCBfGKeYHKs2BZ5EESM7UO/NwaDG5EJJmfJZ9kzK6l+lHmo8kbJhQj"
    "soodMxF77HTIsj5yfstsNjGHzA0l2Jq9uzi7Hk3GGrp+Gc0GijgL8kAqNhm9JlaZH9xLYMqyIiVGFOMrUSgViNiJk0DJIcSiDWcZ"
    "FtmQLbNgtc5jqdSQ6OeZUPmQpcFGhiKXJiRg7MSasC5zPUHvFROuW2TC3bLEB/mN9Njr8XgURGIl2VLk7hrs8WWyScMkV6ZGeMxY"
    "JsNALIMwyLfAKFaZiBRLodJKqC3jX7Dz2Tl7xc5m5+WxE8auA3U3miUAIfS78LT79y/HR8Py6Q09nf10fTkzB4PrImZpJlORSccT"
    "ubDSLdjNSF5YO56CAmPpNl9DqqyIHQDmJwR04PNnsC88xaxXAAp84ebqlSsg6ZA9gBfZ38mkKkJ8a4zOX/Mgkt+8epbcaJQlSc4A"
    "na9f5UmLZ2AYxiCI0gQOK7IVDihZv/+ikngAP4sYRJNEgVU79TukxP9PSdyciYsI5OCLcVovpXAwLOAv9eq1SORkuTBYDtpHq1CS"
    "G2erlWHuw0EKetJowrxky02iCJJWsPz7Ifv23Y83Qza7vL7+6er23SW93Jz/fH797vbdOZ7fnl+c/fT+1rm+vLwdsquz9+e3t+cN"
    "yAdndvn+8vpmODhknjRLfpFu7pAK4dgq8fNIbIZM3ouwgD6GXe9zlkEMoAwO47iVY8FdBp70WZgIzwmTVZArru1rlm7yEORrqM2i"
    "/XIDbmroYK3ArTj9ZAyZCMPkwUkD9y6U9oUIlTRJLUrcS2/a8P5xyD4Miet7GQsKMbuEmBslMmNhIS63qeSg6YNo/uUpkkMNI5Yy"
    "7MAEcU6becYrgBaxsTA1UaGUhBk+WmotUpCzGZ+M8Rmy702dZEAHrBM5aMdNijjn4DAK4lDGq3xtAwyH6EyJcAaWH5/0I2WhmNJe"
    "EHet20p7UHm+0SYt55HOP/1PKqRP4D8rNfuT3dUvsQgHJSmzTGwd+a9ChHxHnzCKOd3zr0wgi7Kf4UjyPMuSjD/DOXsAk5HwZCmv"
    "QJLzfZnJOGeQ2L1LE9iJGNHk8IByYhlmj95sTugWf8gdmoOVbauzHQu/HrI9IwfKD2IkLF6Bm9rsJbJM5kUWV8456/pnFR3EFPeD"
    "1ZAlBbIoIagUhkUrp8rihGKLzQojLdMhfHOsarNXGkvjFWztpYE9eQMOl6gXDjwPtcM2NCbj91F4vvGZk0hFlhsmSrNcB7iHwut0"
    "kgGnNJEO2ZbSZR5KqophklVipSjZ9l7u4ARfxdXGoszH5xBhAgeqv43RyKhQ2VUqmxvuWmRuIkLytPDBnvRRgNZ8OmRjbFaPJaIG"
    "j/5P4ZGuhW192ZxWrshzmT2DQNmTY/YF+2rMXtbLxwtosXm2kCm52SPSopY512qxoXT98PTPGDXafqwza6mL+QssvlhMrVP/Cfxu"
    "tIvbxg/Uy7SNhNFL3tsK6HIJ94WzN22FRhBENiddgrFt+1JbMRJBzCsbiRQmqsujdZatiggxd0VvGfekcrNAh6rtOF7iOs6QElak"
    "Vea4IYLHbg6j43vbHvhOhulFDVqpBPnD8xxREeFGWbvJkaUvUPPtfhVbA4VtlD1hElL/iDLqseV2tzExnsWvpPSAn6LfRgJpKZ0e"
    "N/gBQr0Y9bDP9GM1/mxFDg0yWmAipHgbLZkLtaEts4qUugj+aCByikxaCFNjyiYTGMHw0TVaKvgkaYUWxEYqa5UFHhZuswIhZNCb"
    "pR0Va9bx8VNJA7qvmqaqQ2J2v3RzYsgilZYHKODtBphCv2ypHusmx4qTB173OVaRu6aFiuDTCn9x9OEoOvJuj747+uHo5uML88mo"
    "MwoRLpNJJaFqt6zozgsQTYJSt7JLieQGPbCT3OnXkrcDaRJY9xuIwaBsIK9g/bZwsE3Tx+qKR6pEqmbaasoq83HyQIzOH41Zcw76"
    "dKHgm+osXtHKvHzZhGPV+iC9u4u5wpQzWZhU1556dYZKtrtTr/WiosW2K1uUrkGR0nIOm3nWW/jtBXxFcuLSPABn5YnjqvsqaVeG"
    "2wHBPnw7wGS1qSp9o60kCSuH3plvRM4kFNuor9TVFfUj4ylrFPCxI/YBydo6fzVXVHCptCJVIZfCllDk/JAmDyqOFIzDeRBK6pnQ"
    "MPHOrmk2MmGummqZFCWE43EzLCF2X5PtYy9w8UpESLht11+ofvdFrk/bRD3DfhIhg9HXTU4S6Fii9GFaUHtUYBGIOPH3wTQtOD11"
    "Cnw0GRL10oSUrJ1VlhQp3AoJuXomL6TqtuirE0nZRG05pFUExRIoHJwkBTfrHQKWSFMZe1yD6ipECb5BoOdl0NXbC93rUVczZujV"
    "JImsUlRkXhHa03rbHjVS1AR3Lc2pK1yVDMwrrS6I1ofmzbQi1DI0CwHqaekGKWlAc7moTZzCaZF9dwIE8ftsuOrq+VRqdMjKV8J7"
    "ZaFHi5Cbq/69xHw4pPSWE0nM6K46FFKEQBVRJOBRdoVqPm+ZQj45a2uvrtozmIme/yFS+kJ111CegDPey9skfU/F2zg8jTFmVHM5"
    "HaoGc32eJnNjsYD60y3vMWahr0IXy481A17QsDDX/S7ZLNLL3DU7Ydhxp8VBfCcNvm/JlJDzEE5R4RSEs3WYwyhPeyw6mJEK5ezR"
    "qA+1kEgh/cUaukcFdkGbKOnAN/bY+sqg5FQxzF32jc2srypvfE4RPaYPuUy994yvpBkaDV4DASdi5tQkTHQPFK9490ST3S50IS2T"
    "XH35Uyaqcr4iMYwZTcwQ6QuIRA7IHtWT8Xzl0d2oPrlzDUA5+/k0tKiLOXLYRmdI6m5UsdRMcSQ8uBnVenQxNp/AR06t11XK0G6A"
    "maBKfED/KUg5oUF8PpMezc7YuNTt6MaqVMBraLVOHvwwkJkqFYe+gczv0G2Pyjt9Rc0EEFRNec3EMp0bWJV6VC1VszOvYlf37L5w"
    "ZW9okN5KHpwiXveJ3smt1qjxsEbYglkKW1ek+rsMFvSS071mgu5UcQ4cAsNif4jGPjHmaOr8wETUyddaBGeTB+6d4qhrK0kW+wuq"
    "bOlIu6C8Hjdu6A4WbCCvqzKvtaODNT4267FGqvl40ZltqEFPQ9mbVJrx5Aexqe+TdFGrBkGjg2zSQ3ahO/B+aYf5qd1Hxg+3+g6z"
    "LewdUjoftGNQZ/4lUCDn9eX1b7/+Z+d6+mvqZ7V7sXXgYUmbhS4t+wy3Izz5/7DOCHXAGvvhfDw9eHVrfT7GEGAn3RhD0nyzE2G9"
    "4LL8UGC42Qvljq8dHN1fUncBMcrEstsl+E2R04lm7zZxXqf4z2l51LkK75D/ui73neb080ruHD6g55Np/667p+Bd9TaKfYPmraPX"
    "w03EtNN+lcjL60+arbq3oJ2Lje7NRP9QfVOwr8oqQO0/ZozeJUMdwrXwnaggpTAu6eqNUQSZ/fuBBk0oV9TcddE6v2tPQv7br/+u"
    "yf4fNu3qr7ZqJOLAl4pm10cNh76PGhGdFBw90U9Z057TBloLLJU35EaX+LR3Yb6XV432FyGAdq/Tu9ehU/1zgb5/VbxdNw/gE6tV"
    "JleiGjUNDK2jzhBSNZmM5hE9s35fDaxfsxzjsqe0/g7Pbd2fbiyjHMzqtoR+D6mVZhGvhmnp31WcXG5yrrn3iihVvIYq25Y4t49R"
    "F2DKxENjYhtF7o/eGN02xrgubw+mhr61pBsk+KXj0BWi45B7Go5D90mOY0wr69Hl0uC/UEsDBBQAAAAIALQYR10TZbK2cwAAAI8A"
    "AAAgAAAAcGFydF8zL2NpZmFyMTBjL3JlcXVpcmVtZW50cy50eHQljdsOgyAQRN/3WwjpgtI+wP4LFdOSyEVBG/9esI9ncmYm7iGf"
    "ZJALxbSEMvk/IjItINvobCEjeMMBgq15SXXxbzKSv3pU0zZ9uyB6/abDF58imQfHJ9MI27zuc6n3jMSufVz6NWFkWkFdXSAzcNX+"
    "R7gAUEsBAhQDFAAAAAgAixhHXeRcZ47YBgAAVhAAABkAAAAAAAAAAAAAAKSBAAAAAHBhcnRfMy9jaWZhcjEwYy9jb21tb24ucHlQ"
    "SwECFAMUAAAACACXGEddzr4/qy8LAABRHAAAHwAAAAAAAAAAAAAApIEPBwAAcGFydF8zL2NpZmFyMTBjL3ByZXBhcmVfZGF0YS5w"
    "eVBLAQIUAxQAAAAIAKcYR13JP7+jYAsAACscAAAcAAAAAAAAAAAAAACkgXsSAABwYXJ0XzMvY2lmYXIxMGMvcnVuX3BhcnQzLnB5"
    "UEsBAhQDFAAAAAgAtBhHXRNlsrZzAAAAjwAAACAAAAAAAAAAAAAAAKSBFR4AAHBhcnRfMy9jaWZhcjEwYy9yZXF1aXJlbWVudHMu"
    "dHh0UEsFBgAAAAAEAAQALAEAAMYeAAAAAA=="
)
if os.path.exists("neural_network_confidence/part_3/cifar10c/run_part3.py"):
    print(f"Using part_3/cifar10c from branch '{BRANCH}'.")
else:
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(PART3_CODE_ZIP))) as z:
        z.extractall("neural_network_confidence")
    print("Installed the embedded Part 3 code (local to this Colab session).")
os.chdir("neural_network_confidence/part_3/cifar10c")

ROOT = "artifacts"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/nnc_part3_artifacts"
os.makedirs(ROOT, exist_ok=True)
print("Code:", os.getcwd(), "| artifacts:", ROOT)

In [ ]:
!pip -q install "gdown>=5,<6"
!python prepare_data.py --root "$ROOT"

In [ ]:
!python run_part3.py --root "$ROOT"

In [ ]:
import glob, shutil, pandas as pd
from IPython.display import Image, display
RESULTS = sorted(glob.glob(f"{ROOT}/results/part3_*/"))[-1]
display(pd.read_csv(RESULTS + "part3_summary.csv").round(4))
for name in ("part3_boxplots", "part3_reliability", "part3_risk_coverage"):
    display(Image(RESULTS + "figures/" + name + ".png", width=1000))
zip_path = shutil.make_archive(RESULTS.rstrip("/"), "zip", RESULTS)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(zip_path)